[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C28_Frontier_Diffusion_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy 在**玩具规模**（2D 分布、小图、小网络）复现驱动 SD3/Flux/SoRA 的前沿机制，再与朴素参考 **对拍**。

这个 notebook 做四件事：① 确认环境；② 造一个贯穿全课的 2D 玩具数据分布；③ 复习一行扩散闭式加噪（基础在 C16），确认我们站在同一起点；④ 立下全课的纪律——**对拍（differential testing）**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画 2D 分布/采样轨迹）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 贯穿全课的玩具数据：2D 分布

真实模型生成 512×512×3 的图（百万维）。本课把它换成 **2D 平面上的分布**——维度小到能 `print`、能画散点，但「学一个分布、从噪声采样它」的机制完全一样。

先造两个经典的 2D 玩具分布：**八高斯**（8 个环形排列的高斯团）与 **双月**。后面模块反复用它们当「数据」。

In [ ]:
def sample_eight_gaussians(n, std=0.08, radius=1.0, rng=None):
    '''8 个均匀排布在半径 radius 圆周上的高斯团。返回 (n,2)。'''
    rng = rng or np.random.default_rng(0)
    centers = np.array([[radius*np.cos(2*np.pi*k/8), radius*np.sin(2*np.pi*k/8)] for k in range(8)])
    idx = rng.integers(0, 8, size=n)
    return centers[idx] + std * rng.standard_normal((n, 2))

def sample_two_moons(n, noise=0.06, rng=None):
    '''两个交错的半月。返回 (n,2)。'''
    rng = rng or np.random.default_rng(0)
    n1 = n // 2; n2 = n - n1
    t1 = np.pi * rng.random(n1)
    m1 = np.stack([np.cos(t1), np.sin(t1)], 1)
    t2 = np.pi * rng.random(n2)
    m2 = np.stack([1 - np.cos(t2), 1 - np.sin(t2) - 0.5], 1)
    x = np.concatenate([m1, m2], 0) + noise * rng.standard_normal((n, 2))
    return x

rng = np.random.default_rng(0)
X8 = sample_eight_gaussians(2000, rng=rng)
Xm = sample_two_moons(2000, rng=rng)
print('八高斯 shape', X8.shape, '范围', np.round(X8.min(0),2), '~', np.round(X8.max(0),2))
print('双月   shape', Xm.shape, '范围', np.round(Xm.min(0),2), '~', np.round(Xm.max(0),2))
assert X8.shape == (2000, 2) and Xm.shape == (2000, 2)
# 八高斯应当有 8 个聚类中心：用最近中心计数检验大致均匀
centers = np.array([[np.cos(2*np.pi*k/8), np.sin(2*np.pi*k/8)] for k in range(8)])
nearest = np.argmin(((X8[:,None,:]-centers[None])**2).sum(-1), axis=1)
counts = np.bincount(nearest, minlength=8)
print('八高斯各团样本数:', counts)
assert (counts > 100).all(), '八个团都应有可观样本'
print('✅ 玩具数据就绪：后面模块用它们当「待生成的分布」')

## 3 · 复习一行：扩散闭式加噪（基础见 C16）

本课默认你已懂扩散前向：从 $x_0$ 直接跳到第 $t$ 步的闭式是

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\quad \epsilon\sim\mathcal N(0,I).$$

确认我们站在同一起点：实现它，并验证 **闭式一步加噪 == 逐步加噪**（C16 的核心恒等式）。

In [ ]:
def make_schedule(T=100, beta_min=1e-4, beta_max=0.02):
    betas = np.linspace(beta_min, beta_max, T)
    alphas = 1.0 - betas
    abar = np.cumprod(alphas)          # ᾱ_t
    return betas, alphas, abar

betas, alphas, abar = make_schedule(T=100)

def q_sample_closed(x0, t, eps, abar):
    '''闭式：一步从 x0 跳到 x_t。t 为 0-based 步索引。'''
    a = abar[t]
    return np.sqrt(a) * x0 + np.sqrt(1 - a) * eps

def q_sample_stepwise(x0, t, eps_each, betas):
    '''逐步：用同一批底噪 eps_each 一步步加到 x_t（验证用）。'''
    x = x0.copy()
    for s in range(t + 1):
        x = np.sqrt(1 - betas[s]) * x + np.sqrt(betas[s]) * eps_each[s]
    return x

# 闭式与逐步在「同一组底噪」下未必逐点相等（逐步是多个独立噪声的叠加），
# 但二者的边际分布相同：均值 √ᾱ·x0、方差 (1-ᾱ)。我们验证边际统计一致。
x0 = sample_eight_gaussians(20000, rng=rng)
t = 50
eps = rng.standard_normal(x0.shape)
xt = q_sample_closed(x0, t, eps, abar)
emp_mean = xt.mean(0); emp_var = xt.var(0)
# 理论：E[x_t]=√ᾱ·E[x0]，Var[x_t]=ᾱ·Var[x0]+(1-ᾱ)
th_mean = np.sqrt(abar[t]) * x0.mean(0)
th_var  = abar[t] * x0.var(0) + (1 - abar[t])
print(f't={t}, ᾱ={abar[t]:.3f}')
print('经验均值', np.round(emp_mean,3), ' 理论', np.round(th_mean,3))
print('经验方差', np.round(emp_var,3), ' 理论', np.round(th_var,3))
assert np.allclose(emp_mean, th_mean, atol=0.02)
assert np.allclose(emp_var, th_var, atol=0.02)
print('✅ 闭式加噪的边际统计与理论一致 —— 我们和 C16 站在同一起点')

## 4 · 立纪律：对拍（differential testing）

本课每个机制都要和一个**朴素参考实现**比对，标准是 `np.allclose(impl, ref, atol=...)`。

把它封成一个工具，后面每个模块都用它当统一裁判。先演示：一个「向量化」实现 对拍 一个「循环」参考。

In [ ]:
def check_allclose(name, got, ref, atol=1e-8):
    '''对拍：我的实现 vs 朴素参考。打印并 assert。'''
    got = np.asarray(got); ref = np.asarray(ref)
    ok = np.allclose(got, ref, atol=atol)
    max_err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<28}] allclose={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参考不一致！'
    return ok

# 演示：向量化 pairwise 距离 对拍 双重循环参考
A = rng.standard_normal((5, 2)); B = rng.standard_normal((4, 2))
def dist_loop(A, B):
    D = np.zeros((len(A), len(B)))
    for i in range(len(A)):
        for j in range(len(B)):
            D[i, j] = np.sqrt(((A[i]-B[j])**2).sum())
    return D
D_vec = np.sqrt(((A[:,None,:]-B[None])**2).sum(-1))
check_allclose('vectorized dist vs loop', D_vec, dist_loop(A, B))
print('\n这就是全课的工作流：写机制 -> 对拍朴素参考 -> assert 兜底。')

## 5 · 一个采样器骨架：确定性 ODE 积分

本课模块 03/05 反复用「从噪声出发、沿某个向量场积分到数据」的采样。先把最简单的 **Euler 积分器** 写出来——给定一个向量场 `field(x, t)`，从 `x0` 沿时间走 `n_steps` 步。这是后面 flow/consistency 采样的公共底座。

In [ ]:
def euler_integrate(field, x0, t0, t1, n_steps):
    '''对 dx/dt = field(x,t) 做 Euler 积分，从 t0 到 t1，返回终点。'''
    x = x0.copy().astype(float)
    dt = (t1 - t0) / n_steps
    t = t0
    for _ in range(n_steps):
        x = x + dt * field(x, t)
        t += dt
    return x

# 自检：恒定速度场 v=const，Euler 应精确（直线运动）
v_const = np.array([1.0, -0.5])
x_end = euler_integrate(lambda x, t: np.broadcast_to(v_const, x.shape),
                        np.zeros((3, 2)), 0.0, 1.0, n_steps=10)
assert np.allclose(x_end, v_const), '恒速场积分 1 个单位时间应位移 = v'
print('Euler 积分恒速场 ->', x_end[0], '（应=[1,-0.5]）')
print('✅ 采样器底座就绪：直线运动被精确积分（直线路径 = 少步采样的关键，见模块 03/05）')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 玩具上写出的每个前沿机制（latent 两阶段、patchify/adaLN、直线速度场、CFG 外推、一致性自洽损失），都会用 `np.allclose` 对拍朴素参考；结构正确则数值一致，数值一致则逻辑可迁移到 SD3/DiT/Flux 的真实实现。

**接下来五个模块**：01 latent 扩散 → 02 DiT → 03 flow matching → 04 CFG → 05 consistency。前两块讲「在哪生成、用什么生成」，中间讲「怎么学、怎么控」，最后讲「怎么压成一步」。

下一站：**模块 01 · Latent Diffusion 与 Stable Diffusion**。